# RQ3 Faithfulness Drift Investigation

Results section flags a small but statistically significant negative drift in
mean faithfulness across the 20 repeated runs (slope -0.002/run, r=-0.519,
p=0.019). This notebook tests the three candidate explanations named in the
results text: prompt order effects, host level load across the session, and
whether the drift is broad based or driven by one volatile sample. Sampling
drift (irreducible judge stochasticity) is the default explanation if the
other two are ruled out and the effect is broad based rather than
concentrated.

Data source: `results/rq3/data/rq3_raw.csv` and
`results/rq3/data/rq3_run_summary.csv` in the pipeline repo, the same files
used for the ICC and drift analysis reported in Chapter 5.


In [1]:
import pandas as pd
from scipy import stats
from pathlib import Path

_HERE = Path.cwd()
RAW_CSV = (_HERE / "../../../results/rq3/data/rq3_raw.csv").resolve()
SUMMARY_CSV = (_HERE / "../../../results/rq3/data/rq3_run_summary.csv").resolve()

raw = pd.read_csv(RAW_CSV)
summary = pd.read_csv(SUMMARY_CSV)
raw.head()


,run_id,sample_id,question,answer,elapsed_seconds,faithfulness,answer_relevance,context_precision,context_recall
0,1,1,What are the global implications of the USA Su...,The global implications of the USA Supreme Cou...,75.84,0.619048,0.940646,1.000000,1.0
1,1,2,Which companies are the main contributors to G...,"According to the Carbon Majors database, the m...",94.19,0.041667,0.816346,0.333333,1.0
2,1,3,Which private companies in the Americas are th...,"According to the Carbon Majors database, the l...",49.28,0.333333,0.978856,0.333333,1.0
3,1,4,What action did Amnesty International urge its...,Amnesty International urged its supporters to ...,29.83,0.400000,0.845581,0.500000,1.0
4,1,5,What are the recommendations made by Amnesty I...,Amnesty International made several recommendat...,65.74,0.047619,0.961302,0.500000,1.0


## Hypothesis 1: prompt order effects

Prompt order can only explain a run to run drift if the order in which
samples are submitted changes between runs. If `sample_id` is presented in
the same fixed sequence in every run, order is a constant across runs and
cannot produce a trend that only shows up when runs are compared.

In [2]:
orders = raw.groupby("run_id")["sample_id"].apply(lambda s: tuple(s.tolist()))
n_distinct_orders = orders.nunique()
print(f"Distinct sample orders across the 20 runs: {n_distinct_orders}")


Distinct sample orders across the 20 runs: 1


Result: every run presents the 20 samples in the same fixed order. Prompt
order is a constant across the session, so it cannot be the mechanism behind
a run indexed drift. **Ruled out.**

## Hypothesis 2: host level load across the session

`elapsed_seconds` (per sample generation and evaluation latency) is the only
host level load proxy recorded per run. If host load builds up across the
session (thermal throttling, resource contention), the per run mean latency
should trend upward with `run_id`, and that latency should predict
faithfulness.

In [3]:
run_elapsed = raw.groupby("run_id")["elapsed_seconds"].mean().reset_index()
merged = run_elapsed.merge(summary[["run_id", "mean_faithfulness"]], on="run_id")

latency_vs_run = stats.linregress(merged["run_id"], merged["elapsed_seconds"])
latency_vs_faithfulness = stats.linregress(merged["elapsed_seconds"], merged["mean_faithfulness"])

print("elapsed_seconds vs run_id: "
      f"slope={latency_vs_run.slope:.5f}, r={latency_vs_run.rvalue:.3f}, p={latency_vs_run.pvalue:.3f}")
print("elapsed_seconds vs mean_faithfulness: "
      f"slope={latency_vs_faithfulness.slope:.5f}, r={latency_vs_faithfulness.rvalue:.3f}, p={latency_vs_faithfulness.pvalue:.3f}")
merged


elapsed_seconds vs run_id: slope=-0.01056, r=-0.103, p=0.667
elapsed_seconds vs mean_faithfulness: slope=-0.00313, r=-0.087, p=0.715


,run_id,elapsed_seconds,mean_faithfulness
0,1,56.3525,0.4998
1,2,55.9570,0.5370
2,3,55.9610,0.5480
3,4,56.0285,0.5294
4,5,55.7540,0.5136
5,6,55.6410,0.5557
6,7,54.1845,0.5149
7,8,56.2720,0.5354
8,9,56.0120,0.5372
9,10,55.0565,0.5302


Result: per run mean latency shows no trend across the session (p=0.667)
and does not predict per run mean faithfulness (p=0.715). Host level load,
at least as proxied by generation and evaluation latency, is not a plausible
mechanism. **Ruled out.**

## Hypothesis 3: broad based drift vs single volatile sample

The five samples pinned at an exact value across all 20 runs (`sample_id`
3, 4, 6, 7, 20) contribute zero variance and are excluded here, since a
constant cannot carry a trend. For each of the remaining 15 samples, this
regresses that sample's faithfulness score against `run_id` to check
whether the session level drift is concentrated in one or two samples or
spread thinly across most of them.

In [4]:
PINNED_SAMPLES = {3, 4, 6, 7, 20}

per_sample_trend = []
for sample_id, g in raw.groupby("sample_id"):
    if sample_id in PINNED_SAMPLES:
        continue
    r = stats.linregress(g["run_id"], g["faithfulness"])
    per_sample_trend.append((sample_id, r.slope, r.rvalue, r.pvalue))

trend_df = pd.DataFrame(per_sample_trend, columns=["sample_id", "slope", "r", "p"]).sort_values("p")
n_negative = (trend_df["slope"] < 0).sum()
n_significant = (trend_df["p"] < 0.05).sum()
print(f"Samples with a negative run-indexed slope: {n_negative} / {len(trend_df)}")
print(f"Samples individually significant at p<0.05: {n_significant} / {len(trend_df)}")
trend_df


Samples with a negative run-indexed slope: 10 / 15
Samples individually significant at p<0.05: 1 / 15


,sample_id,slope,r,p
12,17,-0.008499,-0.455936,0.043339
8,13,-0.016165,-0.390861,0.088384
14,19,-0.002197,-0.365439,0.113095
13,18,-0.009345,-0.328901,0.156794
11,16,-0.002914,-0.309095,0.184820
10,15,0.002864,0.302750,0.194471
6,11,0.002747,0.231723,0.325589
2,5,-0.000366,-0.222931,0.344788
7,12,0.000437,0.151375,0.524080
9,14,0.001253,0.149628,0.528937


## Conclusion

Both concrete, testable mechanisms named in the results text are ruled out:
prompt order is constant across runs, and generation/evaluation latency
neither trends across the session nor predicts faithfulness. The drift is
also not attributable to one volatile sample: 10 of the 15 non pinned
samples carry a negative run indexed slope, but only 1 of the 15 reaches
individual significance at p<0.05, meaning the session level trend is a weak,
broadly shared tendency that only becomes detectable once averaged across
all 20 samples per run, not a single outlier sample driving the effect.

This is consistent with the drift being ordinary judge side sampling
stochasticity that happens to correlate weakly with run order over a
20 run session, rather than a systematic artifact of the pipeline (reload,
prompt order, or host load). No mechanism found here elevates it above that
default explanation.
